In [0]:
df = spark.table("workspace.default.cross_data")

In [0]:
df.columns

['journey_id',
 'train_number',
 'train_type',
 'departure_date',
 'year',
 'month',
 'day_of_week',
 'departure_hour',
 'is_weekend',
 'is_night_departure',
 'is_peak_hour',
 'is_festival_season',
 'season',
 'zone',
 'zone_abbr',
 'source_station_category',
 'destination_station_category',
 'distance_km',
 'num_scheduled_stops',
 'scheduled_travel_hours',
 'track_doubled',
 'is_hdn_route',
 'traction_type',
 'is_electrified',
 'psr_count',
 'is_circular_route',
 'is_monsoon_season',
 'is_fog_risk',
 'fog_risk_score',
 'zone_fog_index',
 'zone_congestion_index',
 'season_severity_score',
 'loco_age_years',
 'coach_age_years',
 'has_lhb_coaches',
 'is_rake_shared',
 'maintenance_score',
 'seat_utilisation_pct',
 'is_overloaded',
 'late_incoming_rake',
 'is_special_train',
 'route_historical_ontime_pct',
 'primary_delay_cause',
 'delay_minutes',
 'is_delayed',
 'merge_key_train_number',
 'PNR Number',
 'Train Number',
 'Date of Journey',
 'Class of Travel',
 'Quota',
 'Source Station',


In [0]:
from pyspark.sql.functions import col, when

# Define thresholds (safe defaults)
DELAY_THRESHOLD = 60   # minutes
SHORT_DISTANCE = 200   # km

df = df.withColumn(
    "no_show",
    when(
        (col("Confirmation Status") == "Confirmed") &
        (col("delay_minutes") > DELAY_THRESHOLD) &
        (col("distance_km") < SHORT_DISTANCE),
        1
    ).otherwise(0)
)

# Check distribution
df.groupBy("no_show").count().show()

+-------+------+
|no_show| count|
+-------+------+
|      0|150459|
|      1|  1502|
+-------+------+



In [0]:
from pyspark.sql.functions import col, rand

# Separate classes
df_no_show = df.filter(col("no_show") == 1)
df_not_no_show = df.filter(col("no_show") == 0)

# Count minority
minority_count = df_no_show.count()
majority_count = df_not_no_show.count()

# Safety check (avoid division issues)
fraction = minority_count / majority_count if majority_count > 0 else 0.1

# Downsample majority
df_not_no_show_sampled = df_not_no_show.sample(
    fraction=fraction,
    seed=42
)

# Combine
df_balanced = df_no_show.union(df_not_no_show_sampled)

# Shuffle
df_balanced = df_balanced.orderBy(rand())

# Check distribution
df_balanced.groupBy("no_show").count().show()

+-------+-----+
|no_show|count|
+-------+-----+
|      1| 1502|
|      0| 1544|
+-------+-----+



In [0]:
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col

# Selected columns
feature_cols = [
    "distance_km",
    "delay_minutes",
    "departure_hour",
    "zone_congestion_index",
    "seat_utilisation_pct",
    "is_weekend",
    "is_peak_hour",
    "is_night_departure",
    "is_monsoon_season",
    "is_fog_risk",
    "is_overloaded",
    "late_incoming_rake"
]

categorical_cols = [
    "train_type",
    "Quota",
    "Class of Travel",
    "Booking Channel"
]

# Apply StringIndexer safely
indexers = [
    StringIndexer(inputCol=col_name, outputCol=col_name + "_idx", handleInvalid="keep")
    for col_name in categorical_cols
]

df_indexed = df_balanced

for indexer in indexers:
    df_indexed = indexer.fit(df_indexed).transform(df_indexed)

# Final feature list
final_features = feature_cols + [c + "_idx" for c in categorical_cols]

# Select final dataset
df_final = df_indexed.select(final_features + ["no_show"])

df_final.show(5)

+-----------+-------------+--------------+---------------------+--------------------+----------+------------+------------------+-----------------+-----------+-------------+------------------+--------------+---------+-------------------+-------------------+-------+
|distance_km|delay_minutes|departure_hour|zone_congestion_index|seat_utilisation_pct|is_weekend|is_peak_hour|is_night_departure|is_monsoon_season|is_fog_risk|is_overloaded|late_incoming_rake|train_type_idx|Quota_idx|Class of Travel_idx|Booking Channel_idx|no_show|
+-----------+-------------+--------------+---------------------+--------------------+----------+------------+------------------+-----------------+-----------+-------------+------------------+--------------+---------+-------------------+-------------------+-------+
|      193.0|        126.0|            18|                 0.92|                75.3|         0|           1|                 0|              0.0|        0.0|          0.0|               0.0|           4.0

In [0]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler

# Fill nulls safely
df_clean = df_final.fillna(0)

# Assemble features
assembler = VectorAssembler(
    inputCols=final_features,
    outputCol="features",
    handleInvalid="keep"   # extra safety
)

df_model = assembler.transform(df_clean)

# Select final columns
df_model = df_model.select("features", "no_show")

# Train-test split
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)

# Check counts
print("Train count:", train_df.count())
print("Test count:", test_df.count())

Train count: 2488
Test count: 558


In [0]:
from pyspark.ml.classification import LogisticRegression

# Define model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="no_show",
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0  # pure L2 (stable)
)

# Train
lr_model = lr.fit(train_df)

# Predict
predictions = lr_model.transform(test_df)

# Quick peek
predictions.select("no_show", "prediction", "probability").show(10, False)

+-------+----------+-----------------------------------------+
|no_show|prediction|probability                              |
+-------+----------+-----------------------------------------+
|1      |1.0       |[0.041427472368389694,0.9585725276316103]|
|1      |1.0       |[0.11871856215327231,0.8812814378467277] |
|1      |1.0       |[0.053618272159071285,0.9463817278409287]|
|1      |1.0       |[0.2526299049443728,0.7473700950556272]  |
|1      |1.0       |[0.10622730042679697,0.893772699573203]  |
|1      |1.0       |[0.08520886742662237,0.9147911325733776] |
|1      |1.0       |[0.029215195535395564,0.9707848044646045]|
|1      |1.0       |[0.1407237596751374,0.8592762403248626]  |
|1      |1.0       |[0.05042206243292857,0.9495779375670714] |
|1      |1.0       |[0.040942621386766805,0.9590573786132331]|
+-------+----------+-----------------------------------------+
only showing top 10 rows


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# AUC (very important metric)
evaluator = BinaryClassificationEvaluator(
    labelCol="no_show",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluator.evaluate(predictions)
print("AUC:", auc)

AUC: 0.9946531642960215


In [0]:
predictions.groupBy("no_show", "prediction").count().show()

+-------+----------+-----+
|no_show|prediction|count|
+-------+----------+-----+
|      1|       1.0|  264|
|      0|       0.0|  285|
|      0|       1.0|    9|
+-------+----------+-----+



In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="no_show",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = accuracy_evaluator.evaluate(predictions)

print("Accuracy:", accuracy)

Accuracy: 0.9838709677419355


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Precision
precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="no_show",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

precision = precision_evaluator.evaluate(predictions)

# Recall
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="no_show",
    predictionCol="prediction",
    metricName="weightedRecall"
)

recall = recall_evaluator.evaluate(predictions)

# F1 Score
f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="no_show",
    predictionCol="prediction",
    metricName="f1"
)

f1 = f1_evaluator.evaluate(predictions)

print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Precision: 0.9844026940801134
Recall: 0.9838709677419355
F1 Score: 0.9838807720571413


Decision  Tree

In [0]:
from pyspark.ml.classification import DecisionTreeClassifier

# Define model
dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="no_show",
    maxDepth=5,          # keep small to avoid overfitting
    seed=42
)

# Train
dt_model = dt.fit(train_df)

# Predict
dt_predictions = dt_model.transform(test_df)

# Preview
dt_predictions.select("no_show", "prediction", "probability").show(10, False)

+-------+----------+-----------------------------------------+
|no_show|prediction|probability                              |
+-------+----------+-----------------------------------------+
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
|1      |1.0       |[0.004488330341113106,0.9955116696588869]|
+-------+----------+-----------------------------------------+
only showing top 10 rows


In [0]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Accuracy
accuracy = MulticlassClassificationEvaluator(
    labelCol="no_show",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(dt_predictions)

# Precision
precision = MulticlassClassificationEvaluator(
    labelCol="no_show",
    predictionCol="prediction",
    metricName="weightedPrecision"
).evaluate(dt_predictions)

# Recall
recall = MulticlassClassificationEvaluator(
    labelCol="no_show",
    predictionCol="prediction",
    metricName="weightedRecall"
).evaluate(dt_predictions)

# F1
f1 = MulticlassClassificationEvaluator(
    labelCol="no_show",
    predictionCol="prediction",
    metricName="f1"
).evaluate(dt_predictions)

# AUC
auc = BinaryClassificationEvaluator(
    labelCol="no_show",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(dt_predictions)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("AUC:", auc)

Accuracy: 0.9946236559139785
Precision: 0.9946310648754981
Recall: 0.9946236559139785
F1: 0.9946241580152029
AUC: 0.9980416408987838


In [0]:
dt_predictions.groupBy("no_show", "prediction").count().show()

+-------+----------+-----+
|no_show|prediction|count|
+-------+----------+-----+
|      1|       1.0|  263|
|      0|       0.0|  292|
|      0|       1.0|    2|
|      1|       0.0|    1|
+-------+----------+-----+



In [0]:
import joblib

In [0]:
joblib.dump(dt_model, "dt_model.pkl")

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-8066547515798784>, line 1
----> 1 joblib.dump(dt_model, "dt_model.pkl")

File /databricks/python/lib/python3.12/site-packages/joblib/numpy_pickle.py:553, in dump(value, filename, compress, protocol, cache_size)
    551 elif is_filename:
    552     with open(filename, 'wb') as f:
--> 553         NumpyPickler(f, protocol=protocol).dump(value)
    554 else:
    555     NumpyPickler(filename, protocol=protocol).dump(value)

File /usr/lib/python3.12/pickle.py:481, in _Pickler.dump(self, obj)
    479 if self.proto >= 4:
    480     self.framer.start_framing()
--> 481 self.save(obj)
    482 self.write(STOP)
    483 self.framer.end_framing()

File /databricks/python/lib/python3.12/site-packages/joblib/numpy_pickle.py:355, in NumpyPickler.save(self, obj)
    352     wrapper.write_array(obj, self)
    353     return
--> 355 return 